# Colab SDK analytics verification (dev only)
Tests the real Colab background worker and cookie bridge. The success-path fixtures
replace Engine/Scoreboard work: no paid evaluation or public submission is made.
Four test events are sent to screamingface-dev after explicit acceptance.
Dev's 90-day deletion policy is not yet enforced. Disabling does not delete history.
Run in Chrome and Safari. The known first-run Colab JavaScript warning may still appear.


In [ ]:
%pip install -q "screamingface[notebook] @ git+https://github.com/ScreamingFace/screamingface.git@OME-1124-local-sdk-analytics#subdirectory=packages/screamingface"


In [ ]:
import os
# Set before analytics starts; restart the runtime if it was already used.
os.environ["SCREAMINGFACE_ANALYTICS_ENDPOINT"] = "https://analytics.dev.screamingface.ai"
import time
import screamingface as sf
from screamingface._analytics import prompt

# Exercise the same setup hook used by sf.connect(), without starting an Engine.
prompt.offer()
print(sf.analytics.status())


Run the next cell to explicitly consent to these dev analytics test events.
For the cross-runtime check, first inspect status after setup: a remembered acceptance
should recover the same browser ID without running enable again.


In [ ]:
sf.analytics.enable()
deadline = time.monotonic() + 15
while sf.analytics.status()["pending"] and time.monotonic() < deadline:
    time.sleep(0.1)
state = sf.analytics.status()
assert state["enabled"], state
import hashlib
print({"enabled": state["enabled"], "scope": state["scope"],
       "id_fingerprint": hashlib.sha256(state["persistent_id"].encode()).hexdigest()[:16]})


In [ ]:
from unittest.mock import patch
from screamingface._evaluation import url4
from screamingface._scoreboard import leaderboards
from screamingface._analytics import colab

class Report:
    ok = True
class Candidate:
    run_id = "private-test-fixture"

# Observe acknowledgements while retaining the real eval_js/browser exchange.
acknowledgements = []
event_ids = []
original = colab.exchange

def observed(command, event=None):
    result = original(command, event)
    if command == "events":
        acknowledgements.append(result)
        event_ids.append(event["event_id"])
    return result

with patch.object(colab, "exchange", observed),      patch.object(url4, "evaluate_url4_sync", return_value=Report()),      patch.object(leaderboards, "_submission", return_value={}),      patch.object(leaderboards, "prepare_submission_notice", return_value=None),      patch.object(leaderboards, "display_submission_notice", return_value=None),      patch.object(leaderboards, "_sync_json", return_value={}),      patch.object(leaderboards, "_decode_score", return_value=object()):
    with sf.Client(engine_url="http://localhost:9108") as client:
        client.evaluate("fixture")
        client.leaderboards.submit(Candidate())
        deadline = time.monotonic() + 15
        while len(acknowledgements) < 4 and time.monotonic() < deadline:
            time.sleep(0.1)
        assert len(acknowledgements) == 4, acknowledgements
        assert all(r.get("status") == "upstream_accepted" for r in acknowledgements), acknowledgements
        sf.analytics.disable()
        deadline = time.monotonic() + 10
        while sf.analytics.status()["pending"] and time.monotonic() < deadline:
            time.sleep(0.1)
        assert sf.analytics.status()["choice"] == "declined", sf.analytics.status()
        client.evaluate("after-opt-out")
        client.leaderboards.submit(Candidate())
        time.sleep(2.5)
        assert len(acknowledgements) == 4
print({"passed": True, "event_ids": event_ids, "after_opt_out_events": 0})


Provide the four event IDs for a PostHog read-back check. A successful HTTP
acknowledgement alone is not a read-back verification. The last cell leaves consent
declined; a new runtime should remember that choice. To check accepted-ID continuity,
run only through the acceptance cell before restarting the runtime/browser.
